In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))



In [3]:
import torch
from torch import nn

class DiscriminatorCNNVersion1(nn.Module):
    # CNN network that combines temporal data (log spectrogram)
    # and summarized audio data such as MFCCs, zero crossings and so forth at the linear layer.

    def __init__(self, handcrafted_features=68):

        super().__init__()  # inherit nn module

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1)  # first layer convolution

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(kernel_size=2)  # reduce dimensionality

        self.flatten = nn.Flatten(start_dim=1)

        # input is 16, 4x4, feature maps for each input
        self.hidden = nn.Linear(16*64*703 + handcrafted_features, 64)

        # output of hidden is.. a 64 dim vector, we output a logit!
        self.out = nn.Linear(64, 1)

    def forward(self, logSpec, compressedFeatures):
        # input -> conv -> activations -> pool -> reduce adaptive pool -> flatten  -> hidden layer -> output
        x = self.conv1(logSpec)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)  # (Batch size, 256)

        # combine x and compressed non-temporals
        x = self.hidden(torch.cat([x, compressedFeatures], dim=1))
        x = self.relu(x)  # hidden layer stuff goes through relu again

        return self.out(x)

In [5]:
class SpectrogramOnlyCNN(DiscriminatorCNNVersion1):
    def __init__(self):
        super().__init__(handcrafted_features=0)

    def forward(self, log_spec):
        x = self.pool(self.relu(self.conv1(log_spec)))
        x = self.flatten(x)
        x = self.relu(self.hidden(x))
        return self.out(x) # alt version

In [6]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from utils import features as f

torch.manual_seed(42)

EPOCHS = 10
BATCH_SIZE = 8
THRESHOLD = 0.5

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

data_dir = root / "dataset_clips"


def load_split(filename):
    # I want to:
    # 1. Load the CSV file into a pandas DataFrame.
    # 2. Extract the features and labels from the DataFrame.
    # 3. Convert the audio files to spectrograms and extract the handcrafted features.
    # 4. Return the spectrograms, handcrafted features, and labels as PyTorch tensors.
    df = pd.read_csv(data_dir / filename)  # load it in

    spectrograms = []
    handcrafted_features = []
    labels = []

    for index, row in df.iterrows():

        y, sr = f.load_audio(data_dir / row['audio_path'])

        log_spectrogram = f.extract_log_mel_spectrogram(y, sr)

        mfcc = f.extract_mfcc(y, sr)
        chroma = f.extract_chroma(y, sr)
        spec_centr = f.extract_spectral_centroid(y, sr)
        zcr = f.extract_zero_crossing_rate(y, sr)

        mfcc_compress = f.compress_features(mfcc)
        chroma_compress = f.compress_features(chroma)
        spec_centr_compress = f.compress_features(spec_centr)
        zcr_compress = f.compress_features(zcr)

        combined = np.concat(
            (mfcc_compress, chroma_compress, spec_centr_compress, zcr_compress))

        labels.append(row["label"])

        spectrograms.append(log_spectrogram)
        handcrafted_features.append(combined)

    spectrograms = torch.from_numpy(
        np.stack(spectrograms)).unsqueeze(1).float()
    handcrafted_features = torch.from_numpy(
        np.stack(handcrafted_features)).float()
    labels = torch.tensor(labels, dtype=torch.float32)

    return spectrograms, handcrafted_features, labels

/Users/sambarati/Documents/GitHub/DT2470_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [7]:
from copy import deepcopy
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# features
train_spec, train_hand, train_y = load_split("train.csv")
valid_spec, valid_hand, valid_y = load_split("valid.csv")
test_spec, test_hand, test_y = load_split("test.csv")

# normalize
spec_mean = train_spec.mean()
spec_std = train_spec.std(unbiased=False).clamp_min(1e-6)
hand_mean = train_hand.mean(dim=0)
hand_std = train_hand.std(dim=0, unbiased=False).clamp_min(1e-6)


def make_loader(spec, hand, labels, shuffle=False):
    spec = (spec - spec_mean) / spec_std
    hand = (hand - hand_mean) / hand_std
    return DataLoader(
        TensorDataset(spec, hand, labels),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )

train_loader = make_loader(train_spec, train_hand, train_y, shuffle=True)
valid_loader = make_loader(valid_spec, valid_hand, valid_y)
test_loader = make_loader(test_spec, test_hand, test_y)



In [8]:
device = torch.device("cpu")
model = DiscriminatorCNNVersion1().to(device)
loss_function = nn.BCEWithLogitsLoss() # tillåter att skicka in råa logits
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
def forward_batch(model, spectrogram, handcrafted):
    if isinstance(model, SpectrogramOnlyCNN):
        return model(spectrogram)
    return model(spectrogram, handcrafted)

def train(data, model, optim, device, loss_function):
    run_loss, total = 0, 0
    correct = 0

    model.train()
    for spectrogram, handcrafted, labels in tqdm(data, desc=f"Train"):
        spectrogram, handcrafted, labels = spectrogram.to(
            device), handcrafted.to(device), labels.to(device)
        optim.zero_grad()
        output = forward_batch(model, spectrogram, handcrafted).squeeze(-1)
        loss = loss_function(output, labels)
        loss.backward()
        optim.step()

        predicted = torch.sigmoid(output) >= 0.5
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        run_loss += loss.item()*labels.size(0)

    return run_loss/total, correct/total


def evaluate(data, model, loss_function, device):
    model.eval()
    run_loss, total = 0, 0
    correct = 0
    with torch.no_grad():
        for spectrogram, handcrafted, labels in tqdm(data, desc=f"Testing"):
            spectrogram, handcrafted, labels = spectrogram.to(
                device), handcrafted.to(device), labels.to(device)
            output = forward_batch(model, spectrogram, handcrafted).squeeze(-1)
            loss = loss_function(output, labels)
            predicted = torch.sigmoid(output) >= 0.5
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            run_loss += loss.item()*labels.size(0)

    return run_loss/total, correct/total

In [10]:
best_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss, train_acc = train(
        train_loader, model, optimizer, device, loss_function
    )
    val_loss, val_acc = evaluate(
        valid_loader, model, loss_function, device
    )

    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_loss:.4f}, accuracy={train_acc:.1%} | "
        f"val loss={val_loss:.4f}, accuracy={val_acc:.1%}"
    )

Testing: 100%|██████████| 12/12 [00:00<00:00, 22.76it/s]


Epoch 1: train loss=0.9551, accuracy=80.9% | val loss=0.1455, accuracy=94.5%


Testing: 100%|██████████| 12/12 [00:00<00:00, 23.06it/s]


Epoch 2: train loss=0.3318, accuracy=92.2% | val loss=0.8153, accuracy=78.0%


Testing: 100%|██████████| 12/12 [00:00<00:00, 22.24it/s]


...

Epoch 8: train loss=0.0004, accuracy=100.0% | val loss=0.0180, accuracy=100.0%


Testing: 100%|██████████| 12/12 [00:00<00:00, 23.62it/s]


Epoch 9: train loss=0.0003, accuracy=100.0% | val loss=0.0186, accuracy=100.0%


Testing: 100%|██████████| 12/12 [00:00<00:00, 23.22it/s]

Epoch 10: train loss=0.0002, accuracy=100.0% | val loss=0.0195, accuracy=98.9%


In [11]:
model.load_state_dict(best_weights)

test_loss, test_acc = evaluate(
    test_loader, model, loss_function, device
)
print(f"Test loss={test_loss:.4f}, accuracy={test_acc:.1%}")

Testing: 100%|██████████| 18/18 [00:00<00:00, 22.74it/s]

Test loss=0.1545, accuracy=96.4%


In [12]:
y_true, y_prob = [], []

model.eval()
with torch.no_grad():
    for spectrogram, handcrafted, labels in test_loader:
        logits = model(
            spectrogram.to(device),
            handcrafted.to(device),
        ).squeeze(-1)

        y_prob.extend(torch.sigmoid(logits).cpu().numpy())
        y_true.extend(labels.cpu().numpy())

y_true = np.asarray(y_true, dtype=int)
y_prob = np.asarray(y_prob)
y_pred = (y_prob >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else np.nan

metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall / true positive rate": recall_score(
        y_true, y_pred, zero_division=0
    ),
    "F1 score": f1_score(y_true, y_pred, zero_division=0),
    "Specificity / true negative rate": safe_divide(tn, tn + fp),
    "False positive rate": safe_divide(fp, fp + tn),
    "False negative rate": safe_divide(fn, fn + tp),
    "ROC AUC": (
        roc_auc_score(y_true, y_prob)
        if np.unique(y_true).size == 2 else np.nan
    ),
}

display(pd.Series(metrics, name="Test score").round(4))
display(pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
))

Accuracy                            0.9638
Balanced accuracy                   0.9635
Precision                           0.9583
Recall / true positive rate         0.9718
F1 score                            0.9650
Specificity / true negative rate    0.9552
False positive rate                 0.0448
False negative rate                 0.0282
ROC AUC                             0.9878
Name: Test score, dtype: float64

,Predicted 0,Predicted 1
Actual 0,64,3
Actual 1,2,69


In [13]:
torch.manual_seed(42)

spec_model = SpectrogramOnlyCNN().to(device)
spec_optimizer = torch.optim.Adam(spec_model.parameters(), lr=1e-4)

spec_best_loss = float("inf")
spec_best_weights = deepcopy(spec_model.state_dict())

for epoch in range(EPOCHS):
    train_loss, train_acc = train(
        train_loader, spec_model, spec_optimizer, device, loss_function
    )
    val_loss, val_acc = evaluate(
        valid_loader, spec_model, loss_function, device
    )

    if val_loss < spec_best_loss:
        spec_best_loss = val_loss
        spec_best_weights = deepcopy(spec_model.state_dict())

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_loss:.4f}, accuracy={train_acc:.1%} | "
        f"val loss={val_loss:.4f}, accuracy={val_acc:.1%}"
    )

spec_model.load_state_dict(spec_best_weights)

test_loss, test_acc = evaluate(
    test_loader, spec_model, loss_function, device
)
print(f"Test loss={test_loss:.4f}, accuracy={test_acc:.1%}")

Testing: 100%|██████████| 12/12 [00:00<00:00, 24.12it/s]


Epoch 1: train loss=1.1133, accuracy=81.4% | val loss=0.0599, accuracy=98.9%


Testing: 100%|██████████| 12/12 [00:00<00:00, 20.82it/s]


Epoch 2: train loss=0.0880, accuracy=96.7% | val loss=0.1083, accuracy=94.5%


Testing: 100%|██████████| 12/12 [00:00<00:00, 23.14it/s]


...

Epoch 9: train loss=0.0002, accuracy=100.0% | val loss=0.0264, accuracy=98.9%


Testing: 100%|██████████| 12/12 [00:00<00:00, 25.06it/s]


Epoch 10: train loss=0.0002, accuracy=100.0% | val loss=0.0271, accuracy=98.9%


Testing: 100%|██████████| 18/18 [00:00<00:00, 26.94it/s]

Test loss=0.2182, accuracy=92.8%


In [14]:
y_true, y_prob = [], []

spec_model.eval()
with torch.no_grad():
    for spectrogram, handcrafted, labels in test_loader:
        logits = spec_model(spectrogram.to(device)).squeeze(-1)

        y_prob.extend(torch.sigmoid(logits).cpu().numpy())
        y_true.extend(labels.cpu().numpy())

y_true = np.asarray(y_true, dtype=int)
y_prob = np.asarray(y_prob)
y_pred = (y_prob >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else np.nan


metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall / true positive rate": recall_score(
        y_true, y_pred, zero_division=0
    ),
    "F1 score": f1_score(y_true, y_pred, zero_division=0),
    "Specificity / true negative rate": safe_divide(tn, tn + fp),
    "False positive rate": safe_divide(fp, fp + tn),
    "False negative rate": safe_divide(fn, fn + tp),
    "ROC AUC": (
        roc_auc_score(y_true, y_prob)
        if np.unique(y_true).size == 2 else np.nan
    ),
}

display(pd.Series(metrics, name="Test score").round(4))
display(pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
))

Accuracy                            0.9275
Balanced accuracy                   0.9279
Precision                           0.9420
Recall / true positive rate         0.9155
F1 score                            0.9286
Specificity / true negative rate    0.9403
False positive rate                 0.0597
False negative rate                 0.0845
ROC AUC                             0.9859
Name: Test score, dtype: float64

,Predicted 0,Predicted 1
Actual 0,63,4
Actual 1,6,65
